# 4. Evaluation and analysis

*Authors: Lưu Minh Quân, Phạm Tấn Nhật Thịnh, La Anh Hào · data: `m3_model_selection.csv`, `m3_metrics.json`,
`m3_run.json`, `m3_requirements.json`, `m3_hf_export.json`*

This chapter presents model selection across all 14 configurations (4.1), the official MRR@5 evaluation and
slice breakdown (4.2), an honest diagnosis of MRR@5 saturation together with extra experiments including
ablation, LOPO and the NIL-detection curve (4.3), pipeline runtime and the 23/23 requirement checks (4.4),
and the per-member Hugging Face Parquet export counts (4.5).

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import report_data as rd

rd.check()
ms = rd.model_selection()
met = rd.metrics()
run = rd.run_log()
req = rd.requirements()
hfe = rd.hf_export()
feat = rd.feature_report()
ds = rd.data_summary()

## 4.1 Model selection (14 configurations)

Every configuration is evaluated by **5-fold `GroupKFold` CV** grouped by publication on `train` plus MRR@5
on the 8 `valid` papers; configurations are ranked by `choice_score = (cv_mrr5 + valid_mrr5) / 2`.

In [ ]:
t_ms = ms[["config", "kind", "cv_mrr5", "cv_std", "valid_mrr5", "choice_score", "seconds"]].copy()
t_ms.columns = ["Configuration", "Family", "5-fold CV MRR@5", "CV std", "Valid MRR@5", "Choice score", "Seconds"]
rd.table(t_ms, "Table 4.1. Model selection grid across 14 configurations (`m3_model_selection.csv`).",
         {"5-fold CV MRR@5": "{:.6f}", "CV std": "{:.6f}", "Valid MRR@5": "{:.4f}",
          "Choice score": "{:.6f}", "Seconds": "{:.1f}"})

fam_colors = {"baseline": rd.AQUA, "logreg": rd.BLUE, "ranker": rd.ORANGE}
fig = go.Figure()
for fam, sub in ms.groupby("kind", sort=False):
    fig.add_trace(go.Scatter(
        x=sub["seconds"], y=sub["cv_mrr5"], mode="markers+text" if fam == "baseline" else "markers",
        text=["baseline (chosen)"] if fam == "baseline" else None,
        textposition="middle right",
        name=fam,
        marker=dict(size=12 if fam == "baseline" else 9, color=fam_colors.get(fam, rd.BLUE),
                    line=dict(width=1, color=rd.INK) if fam == "baseline" else dict(width=0)),
        customdata=np.stack([sub["config"], sub["valid_mrr5"], sub["choice_score"]], axis=-1),
        hovertemplate="<b>%{customdata[0]}</b><br>5-fold CV MRR@5: <b>%{y:.6f}</b><br>"
                      "Valid MRR@5: %{customdata[1]:.4f}<br>Choice score: %{customdata[2]:.6f}<br>"
                      "Runtime: %{x:.1f} s<extra></extra>",
    ))
fig.update_layout(
    title="5-fold CV MRR@5 vs runtime across all 14 configurations",
    legend=dict(orientation="h", yanchor="top", y=-0.22, xanchor="center", x=0.5),
)
fig.update_xaxes(title_text="Selection runtime (seconds)", range=[0, ms["seconds"].max() * 1.12])
fig.update_yaxes(title_text="5-fold GroupKFold CV MRR@5", range=[0.9988, 1.0001])
rd.show(fig, height=350, margin=dict(l=70, r=25, t=55, b=75))
rd.figure_caption("Figure 4.1. All 14 configurations saturate near 1.0000; `baseline` achieves the highest CV "
                  "score (0.999982) in 31.8 s and is selected with `tau = None` (no assignment needed).")

ch = met["chosen"]
rd.md(
    f"**Chosen configuration:** `{ch['config']}` (`kind = {ch['kind']}`, `tau = {ch['tau']}`). Every family "
    f"reached **1.0000** on both `valid` and `test` (`m3_metrics.json`), and `baseline` also had the highest "
    f"5-fold CV score (**{ms.iloc[0]['cv_mrr5']:.6f}**) at **8–11× lower runtime** ({ms.iloc[0]['seconds']:.1f} s "
    f"vs 244–348 s). Post-processing Hungarian assignment yielded no improvement over `no_assignment = "
    f"{ch['tau_tuning']['no_assignment']:.4f}`, so `tau` was left `None`.")

## 4.2 Main MRR@5 results (`skip` vs `zero`) and slice breakdown

Following Q&A M3 #4, the official metric is **`mrr5_skip`** (`nil_mode_main = "skip"`), averaging $1/	ext{rank}_i$
over entries that have a true arXiv reference. `mrr5_zero` counts unmatched entries as 0 and is shown alongside
it for transparency.

In [ ]:
rows = []
for part in ("train", "valid", "test"):
    for sl in ("all", "manual", "auto", "bib", "bibitem"):
        d = met["final"][part][sl]
        rows.append({
            "Partition": part,
            "Slice": sl,
            "Papers": d["n_papers"],
            "Entries": d["n_entries"],
            "Matched (|Q|)": d["n_with_true_ref"],
            "Matched share": 100 * d["n_with_true_ref"] / d["n_entries"],
            "MRR@5 (skip, main)": d["mrr5_skip"],
            "Hit@1": d["hit1"],
            "Hit@5": d["hit5"],
            "MRR@5 (zero)": d["mrr5_zero"],
        })
t_res = pd.DataFrame(rows)
rd.table(t_res, "Table 4.2. Final model evaluation by partition, label source and bibliography source (`m3_metrics.json`).",
         {"Matched share": "{:.2f}%", "MRR@5 (skip, main)": "{:.5f}", "Hit@1": "{:.5f}",
          "Hit@5": "{:.5f}", "MRR@5 (zero)": "{:.4f}"})

sub_all = t_res[t_res["Slice"] == "all"]
fig = go.Figure()
fig.add_trace(go.Bar(
    x=sub_all["Partition"], y=sub_all["MRR@5 (skip, main)"], name="MRR@5 (skip, official)",
    marker_color=rd.BLUE, text=[f"{v:.5f}" for v in sub_all["MRR@5 (skip, main)"]], textposition="outside",
    hovertemplate="%{x} · MRR@5 (skip): <b>%{y:.5f}</b><extra></extra>",
))
fig.add_trace(go.Bar(
    x=sub_all["Partition"], y=sub_all["MRR@5 (zero)"], name="MRR@5 (zero = matched share)",
    marker_color=rd.ORANGE, text=[f"{v:.4f}" for v in sub_all["MRR@5 (zero)"]], textposition="outside",
    hovertemplate="%{x} · MRR@5 (zero): <b>%{y:.4f}</b><extra></extra>",
))
fig.update_layout(
    barmode="group",
    title="Official MRR@5 (skip) vs MRR@5 (zero) across partitions",
    legend=dict(orientation="h", yanchor="top", y=-0.22, xanchor="center", x=0.5),
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(title_text="Score", range=[0, 1.15])
rd.show(fig, height=340, margin=dict(l=55, r=25, t=55, b=75))
rd.figure_caption("Figure 4.2. Official `mrr5_skip` reaches 1.0000 on `valid` and `test` (0.99998 out-of-fold on "
                  "`train`), while `mrr5_zero` equals the matched-entry share (50.9% / 72.4% / 76.3%).")

rd.md(
    "**Why `mrr5_zero` says nothing about ranking quality.** An entry with no matching arXiv reference always "
    "receives score 0 under `zero`, so `mrr5_zero ≤ |Q| / n_entries`. Because `mrr5_skip = 1.0000` on `valid` "
    "and `test`, `mrr5_zero` equals `318 / 439 = 0.7244` on `valid` and `267 / 350 = 0.7629` on `test`—reflecting "
    "only what fraction of cited works exist on arXiv, not how well the model ranks candidates.")

## 4.3 Saturation analysis and extra diagnostics (Q&A M3 #7)

Why does every tested model—including the unlearned `baseline` and a `title_only` ablation—achieve
**MRR@5 = 1.0000** on `valid` and `test`? The files give four direct, measurable causes:

1. **High identifier availability:** in `m3_feature_report.csv`, **37.03%** of positive pairs have an exact
   arXiv ID match (`f_arxiv_eq`, 0.00% on negatives) and **22.78%** have an exact DOI match (`f_doi_eq`, 0.00%
   on negatives).
2. **Near-verbatim titles:** on positive pairs, mean `f_title_ratio` is **0.9913** (vs 0.3937 on negatives)
   and mean `f_title_tfidf_char` is **0.9887** (vs 0.0971 on negatives).
3. **Compact candidate pools:** each query searches only the citing paper's own arXiv references—a median of
   **18** candidates per eligible paper (`m3_data_summary.json`).
4. **Confirmable positives:** positive labels correspond to pairs confirmable by human labellers or the
   rule matcher (which agrees with manual labels at precision 1.0000 and recall 0.9952).

Per Q&A M3 #7, we therefore complement MRR@5 with **ablation**, **16-paper leave-one-publication-out (LOPO)**,
and the **NIL-detection precision–recall curve** (`nil_detection_valid`), which exposes the genuinely hard
decision boundary of the task.

In [ ]:
ab = pd.DataFrame([
    {"Variant": k, "Valid MRR@5 (skip)": v["valid"], "Test MRR@5 (skip)": v["test"]}
    for k, v in met["ablation"].items()
])
lopo = met["leave_one_publication_out"]
rd.table(ab, "Table 4.3. Five-variant ablation study (`m3_metrics.json`).",
         {"Valid MRR@5 (skip)": "{:.4f}", "Test MRR@5 (skip)": "{:.4f}"})
rd.md(
    f"**Leave-one-publication-out (LOPO)** over **{lopo['n']} manual publications** (excluding the 4 test manual "
    f"papers): mean MRR@5 = **{lopo['mean']:.4f} ± {lopo['std']:.4f}** (1.0000 on every single held-out manual "
    f"paper).")

nil = pd.DataFrame(met["nil_detection_valid"])
nil["share_pct"] = 100 * nil["share_said_nil"]
nil["prec_pct"] = 100 * nil["precision"]
nil["rec_pct"] = 100 * nil["recall"]
op = nil.iloc[(nil["share_said_nil"] - 0.25).abs().idxmin()]

fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.12,
    subplot_titles=("Precision & Recall vs share called NIL", "NIL-detection Precision–Recall curve (valid)"),
)
fig.add_trace(go.Scatter(
    x=nil["share_pct"], y=nil["prec_pct"], mode="lines+markers", name="Precision",
    line=dict(color=rd.BLUE, width=2.2), marker=dict(size=6),
    customdata=nil["threshold"],
    hovertemplate="Called NIL: %{x:.1f}%<br>Precision: <b>%{y:.2f}%</b><br>Threshold: %{customdata:.3f}<extra></extra>",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=nil["share_pct"], y=nil["rec_pct"], mode="lines+markers", name="Recall",
    line=dict(color=rd.ORANGE, width=2.2), marker=dict(size=6),
    customdata=nil["threshold"],
    hovertemplate="Called NIL: %{x:.1f}%<br>Recall: <b>%{y:.2f}%</b><br>Threshold: %{customdata:.3f}<extra></extra>",
), row=1, col=1)
true_nil_pct = 100 * (1 - met["final"]["valid"]["all"]["n_with_true_ref"] / met["final"]["valid"]["all"]["n_entries"])
fig.add_vline(x=true_nil_pct, line_dash="dash", line_color=rd.MUTED, line_width=1.3,
              annotation_text=f"true NIL ({true_nil_pct:.1f}%)", annotation_position="bottom right", row=1, col=1)

fig.add_trace(go.Scatter(
    x=nil["rec_pct"], y=nil["prec_pct"], mode="lines+markers", name="PR curve", showlegend=False,
    line=dict(color=rd.AQUA, width=2.4), marker=dict(size=7),
    customdata=np.stack([nil["threshold"], nil["share_pct"]], axis=-1),
    hovertemplate="Recall: %{x:.2f}%<br>Precision: <b>%{y:.2f}%</b><br>Threshold: %{customdata[0]:.3f}<br>"
                  "Called NIL: %{customdata[1]:.1f}%<extra></extra>",
), row=1, col=2)
fig.add_annotation(
    x=float(op["rec_pct"]), y=float(op["prec_pct"]),
    text=f"P={op['precision']:.3f}, R={op['recall']:.3f}<br>(thr={op['threshold']:.2f})",
    showarrow=True, arrowhead=2, ax=-55, ay=35, row=1, col=2,
)
fig.update_layout(
    title="NIL detection on validation (distinguishing unmatched entries from arXiv matches)",
    legend=dict(orientation="h", yanchor="top", y=-0.22, xanchor="left", x=0.0),
)
fig.update_xaxes(title_text="Entries called NIL (%)", range=[0, 100], row=1, col=1)
fig.update_yaxes(title_text="Score (%)", range=[0, 105], row=1, col=1)
fig.update_xaxes(title_text="NIL Recall (%)", range=[0, 105], row=1, col=2)
fig.update_yaxes(title_text="NIL Precision (%)", range=[0, 105], row=1, col=2)
rd.show(fig, height=370, margin=dict(l=55, r=25, t=65, b=75))
rd.figure_caption("Figure 4.3. NIL-detection threshold sweep on `valid` (439 entries, 121 true NIL = 27.6%). "
                  "Unlike ranking matched entries, deciding whether an entry has *any* arXiv match exhibits a "
                  "real precision–recall trade-off.")

rd.md(
    f"Near the true validation NIL rate ({true_nil_pct:.1f}%), calling **{op['share_pct']:.1f}%** of entries NIL "
    f"(score threshold **{op['threshold']:.3f}**) yields **precision {op['precision']:.3f}** and "
    f"**recall {op['recall']:.3f}**. Reaching 100% recall drops precision to 0.953 at threshold 1.000 and down "
    f"to 0.599 once entries matched purely by title without an ID cross the threshold.\n\n"
    f"> **Proposed harder benchmark (not run):** To stress-test ranking beyond verbatim titles and explicit IDs, "
    f"a future Colab run of `23_model` on `pairs.parquet` could randomly mask identifiers (`arxiv_hint`, `doi`, "
    f"`url`) on 50% of entries and perturb titles (dropping subtitles or injecting typos), comparing `baseline`, "
    f"`logreg` and `LGBMRanker`. Because that experiment requires re-running `23_model` on Colab, it is proposed "
    f"here without fabricating numbers.")

## 4.4 Pipeline runtime and 23/23 requirement verification

The complete Milestone 3 run on **28 Nov 2026** (commit `8c3ee1a`) and the two requirement-audit runs on
commit `689e914` are recorded in `m3_run.json` and `m3_requirements.json`.

In [ ]:
main_run = run["runs"][0]
steps = main_run["steps"]
step_rows = [
    ("20_m3_data (initial build, 27 Nov)", ds["seconds"], "built entries, refs, pubs (20,000 papers)"),
    ("20_m3_data (28 Nov signature check)", steps["data"]["seconds"], "verified signature da32f6a848f6"),
    ("21_labeling", steps["labels"]["seconds"], "validated 20 manual, tuned & ran 3,753 auto papers"),
    ("22_features", steps["features"]["seconds"], "4,821,691 candidate pairs × 25 features"),
    ("23_model", steps["model"]["seconds"], "14 configs, OOF train, ablation, LOPO, predictions"),
    ("26_export_hf", steps["export"]["seconds"], "6 sharded Parquet tables × 4 members"),
    ("24_analysis (28 Nov, commit 689e914)", run["runs"][-1]["seconds"], "23/23 requirement checks PASS"),
]
t_run = pd.DataFrame(step_rows, columns=["Step", "Seconds", "Notes"])
t_run["Duration"] = t_run["Seconds"].map(rd.hm)
rd.table(t_run[["Step", "Duration", "Seconds", "Notes"]],
         f"Table 4.4. Milestone 3 pipeline runtime on Colab CPU (28 Nov full run: {rd.hm(main_run['seconds'])}, "
         f"{main_run['seconds']:,.1f} s).", {"Seconds": "{:,.1f}"})

fig = go.Figure(go.Bar(
    y=t_run["Step"].tolist()[::-1],
    x=t_run["Seconds"].tolist()[::-1],
    orientation="h",
    marker_color=rd.BLUE,
    text=[rd.hm(s) for s in t_run["Seconds"].tolist()[::-1]],
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>%{x:,.1f} s<extra></extra>",
))
fig.update_layout(title="Wall time by Milestone 3 pipeline step (Google Colab CPU)")
fig.update_xaxes(title_text="Seconds", range=[0, t_run["Seconds"].max() * 1.25])
fig.update_yaxes(showgrid=False)
rd.show(fig, height=340, margin=dict(l=230, r=25, t=55, b=45))
rd.figure_caption("Figure 4.4. Per-step wall time on CPU-only Google Colab (`m3_run.json` and `m3_data_summary.json`).")

t_req = pd.DataFrame(req["checks"])[["check", "expected", "actual", "status"]]
t_req.columns = ["Check", "Requirement", "Measured value", "Status"]
n_pass = int((t_req["Status"] == "PASS").sum())
rd.table(t_req, f"Table 4.5. Automated statement and Q&A verification (`m3_requirements.json`: **{n_pass}/{len(t_req)} PASS**).")

## 4.5 Hugging Face Parquet export

`26_export_hf` partitions every table strictly by each member's 5,000-paper ID range with explicit Arrow
schemas and zero out-of-range rows (`m3_hf_export.json`).

In [ ]:
exp = hfe["runs"][-1]
tables_order = ["papers", "references", "elements", "hierarchy", "bibtex", "predictions"]
members = exp["members"]

hf_rows = []
for tbl in tables_order:
    info = exp["tables"][tbl]
    row = {"Table": f"`{tbl}`", "Total rows": info["source_rows"]}
    for m in members:
        md_m = info["members"][m]
        row[m] = f"{md_m['rows']:,} ({md_m['shards']} sh)"
    row["Outside ranges"] = info["papers_outside_ranges"]
    row["Export (s)"] = info["seconds"]
    hf_rows.append(row)
t_hf = pd.DataFrame(hf_rows)
rd.table(t_hf, f"Table 4.6. Exported Parquet rows and shards per member (exported {exp['at']}, "
               f"{exp['seconds']:,.1f} s total; 0 rows outside ranges).", {"Export (s)": "{:.1f}"})

fig = go.Figure()
for i, m in enumerate(members):
    vals = [exp["tables"][t]["members"][m]["rows"] for t in tables_order]
    fig.add_trace(go.Bar(
        x=tables_order, y=vals, name=m, marker_color=rd.SERIES[i],
        hovertemplate=f"{m} · %{{x}}<br><b>%{{y:,}}</b> rows<extra></extra>",
    ))
fig.update_layout(
    barmode="group",
    title="Exported Hugging Face Parquet rows per member and table (log y-axis)",
    legend=dict(orientation="h", yanchor="top", y=-0.22, xanchor="center", x=0.5),
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(type="log", title_text="Rows (log scale)")
rd.show(fig, height=360, margin=dict(l=60, r=25, t=55, b=75))
rd.figure_caption("Figure 4.5. Rows exported into each member's Hugging Face Parquet repository across all six tables.")